# Auditing a published weight edit with TransformerBridge

This notebook examines the Qwen3.5-0.8B edit in Experiment 8 of [*Sparks of Weight-Space Programming*](https://rikiyat.github.io/blog/weight-space-programming/) by Rikiya Takehi, Jayden Teoh, Alex Zhang and Omar Khattab.

We reconstruct the released p50c checkpoint change, fit its dominant direction, and translate the exact weight difference into a hook intervention. The default path uses **CPU, fp32 and at most 32 tokens per input**. The larger bf16 transcript experiment is presented separately as recorded research results.

The [experiment logs and edit artifacts](https://github.com/RikiyaT/lm_understand-logs) are credited to their authors under **CC BY 4.0**. Our reconstruction below uses the published data, rather than executing code from that repository. See [the source license](https://github.com/RikiyaT/lm_understand-logs/blob/4d09ba6d6a7e891dd8063974dfe81b3d78e3494f/LICENSE).

## 1. Pinned inputs and runtime

The first run downloads the checkpoint (roughly 2 GB). The remaining runs reuse the Hugging Face cache. Allow about 6 GiB for the Python process. Time measurements below separate artifact preparation, loading, and forward validation; download time depends on the connection.


In [1]:
# NBVAL_IGNORE_OUTPUT
import time

notebook_started = time.perf_counter()

import hashlib
import io
import json
import platform
import tempfile
import urllib.request
from contextlib import ExitStack, contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn.functional as F
import transformers
from huggingface_hub import snapshot_download
from safetensors import safe_open

from transformer_lens.model_bridge import TransformerBridge
from transformer_lens.utilities import temporarily_swap_parameter

MODEL_ID = "Qwen/Qwen3.5-0.8B"
MODEL_REVISION = "2fc06364715b967f1860aea9cf38778875588b17"
LOGS_REVISION = "4d09ba6d6a7e891dd8063974dfe81b3d78e3494f"
LAYERS = (15, 19)
MAX_TOKENS = 32
torch.set_num_threads(2)
started = time.perf_counter()
timings = {}
print(
    {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "device": "cpu",
        "dtype": "float32",
    }
)


{'python': '3.12.13', 'torch': '2.11.0+cpu', 'transformers': '5.13.0', 'device': 'cpu', 'dtype': 'float32'}


## 2. Reconstruct the published tensors

The p50c artifact stores sparse **replacement values**, not a complete difference matrix. We read the original bf16 tensors, replace the specified entries, and verify both byte hashes against the authors' manifest. Hashing happens before conversion to fp32.

The two small downloaded artifacts also have pinned SHA-256 hashes. Cached artifacts receive the same checks as fresh downloads.


In [2]:
# NBVAL_IGNORE_OUTPUT
ARTIFACT_HASHES = {
    "manifest.json": "83185c1db46e3dafb34af645f8c5ce033c09cc99d4512b9e1b8e3aba9d269225",
    "edit.npz": "e92a92aafd1f1068b4fd94aa0fbe081ba3ddea2047ee7bcac7161d861be299e7",
}
artifact_cache = (
    Path(tempfile.gettempdir()) / "transformerlens-weight-edit-audit" / LOGS_REVISION
)
artifact_cache.mkdir(parents=True, exist_ok=True)


def sha256(data):
    return hashlib.sha256(data).hexdigest()


def read_artifact(filename):
    destination = artifact_cache / filename
    if destination.exists():
        data = destination.read_bytes()
    else:
        url = (
            f"https://raw.githubusercontent.com/RikiyaT/lm_understand-logs/"
            f"{LOGS_REVISION}/exp8_math/runs/p50c/edit/{filename}"
        )
        with urllib.request.urlopen(url, timeout=60) as response:
            data = response.read()
    if sha256(data) != ARTIFACT_HASHES[filename]:
        raise ValueError(f"Artifact checksum mismatch: {filename}")
    if not destination.exists():
        destination.write_bytes(data)
    return data


manifest = json.loads(read_artifact("manifest.json"))
archive = np.load(io.BytesIO(read_artifact("edit.npz")), allow_pickle=False)
snapshot = Path(
    snapshot_download(
        MODEL_ID,
        revision=MODEL_REVISION,
        allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja"],
        max_workers=1,
    )
)
index = json.loads(
    (snapshot / "model.safetensors.index.json").read_text(encoding="utf-8")
)
expected_names = {
    f"model.language_model.layers.{layer}.mlp.down_proj.weight" for layer in LAYERS
}
assert {item["name"] for item in manifest["tensors"]} == expected_names

edits = {}
for number, item in enumerate(manifest["tensors"]):
    assert item["form"] == "sparse" and item["dtype"] == "bfloat16"
    with safe_open(
        str(snapshot / index["weight_map"][item["name"]]), framework="pt"
    ) as reader:
        base = reader.get_tensor(item["name"])
    assert base.dtype == torch.bfloat16 and list(base.shape) == item["shape"]
    base_bits = base.view(torch.uint16).numpy().copy()
    assert sha256(base_bits.tobytes()) == item["base_sha256"]
    indices = archive[f"t{number}_idx"]
    values = archive[f"t{number}_val"]
    assert indices.ndim == values.ndim == 1 and indices.size == values.size
    assert np.unique(indices).size == indices.size
    assert (indices >= 0).all() and (indices < base.numel()).all()
    edited_bits = base_bits.copy()
    edited_bits.reshape(-1)[indices] = values
    assert sha256(edited_bits.tobytes()) == item["edited_sha256"]
    changed_entries = int(np.count_nonzero(base_bits != edited_bits))
    assert changed_entries == item["changed_entries"]
    edited = torch.from_numpy(edited_bits.view(np.int16)).view(torch.bfloat16)
    layer = int(item["name"].split(".layers.")[1].split(".")[0])
    columns = torch.from_numpy(np.unique(indices % base.shape[1])).long()
    assert columns.numel() == item["changed_columns"] == 64
    edits[layer] = {
        "base": base.float(),
        "edited": edited.float(),
        "columns": columns,
        "delta": (edited.float() - base.float())[:, columns],
        "changed_entries": changed_entries,
    }
assert (
    sum(e["changed_entries"] for e in edits.values())
    == manifest["changed_entries_total"]
)
archive.close()
timings["artifacts_and_checkpoint_seconds"] = time.perf_counter() - started
print(
    {
        layer: {
            "columns": len(e["columns"]),
            "changed_bf16_entries": e["changed_entries"],
        }
        for layer, e in edits.items()
    }
)


{15: {'columns': 64, 'changed_bf16_entries': 65260}, 19: {'columns': 64, 'changed_bf16_entries': 64928}}


## 3. Recover the dominant directional update

For HF's output projection, `W` has shape `[d_model, d_mlp]`. Fit the selected columns to

$$
\Delta W_{:,S} \approx -\alpha u (u^\top W_{:,S}), \qquad \|u\|=1.
$$

The fitted directional component is multiplied by `1 - alpha`: alpha=1 deletes it; alpha near 6 reverses and amplifies it. The quantized checkpoint has a small residual, which we measure explicitly. The sign chosen for the singular vector does not change the edit.


In [3]:
# NBVAL_IGNORE_OUTPUT
fit_started = time.perf_counter()
geometry = []
for layer, edit in edits.items():
    delta = edit["delta"].double()
    u, singular_values, _ = torch.linalg.svd(delta, full_matrices=False)
    direction = u[:, 0]
    projection = direction @ edit["base"][:, edit["columns"]].double()
    alpha = -((direction @ delta) @ projection / projection.square().sum()).item()
    fitted_delta = -alpha * torch.outer(direction, projection)
    edit.update(
        direction=direction.float(), alpha=alpha, fitted_delta=fitted_delta.float()
    )
    geometry.append(
        {
            "layer": layer,
            "alpha": alpha,
            "direction_multiplier": 1 - alpha,
            "leading_energy_fraction": (
                singular_values[0].square() / singular_values.square().sum()
            ).item(),
            "relative_fit_residual": (delta - fitted_delta).norm().item()
            / delta.norm().item(),
        }
    )
timings["rank_one_fit_seconds"] = time.perf_counter() - fit_started
display(pd.DataFrame(geometry))


,layer,alpha,direction_multiplier,leading_energy_fraction,relative_fit_residual
0,15,6.000321,-5.000321,0.999995,0.002347
1,19,6.000173,-5.000173,0.999991,0.002944


## 4. Exact weight differences as hooks

If the down projection receives activation `a`, replacing its weights changes its output by

$$
\Delta y = a_{S}\,\Delta W_{:,S}^{\top}.
$$

We first check this identity on a small float64 linear example. Then we validate the **exact difference** on Qwen in fp32, separately from the fitted direction.

The hook pair reads `mlp.hook_post` (the down projection's input) and changes `mlp.hook_out`. Each layer uses activations from the current forward, including earlier interventions.


In [4]:
# NBVAL_IGNORE_OUTPUT
generator = torch.Generator().manual_seed(0)
a = torch.randn(2, 8, generator=generator, dtype=torch.float64)
w = torch.randn(4, 8, generator=generator, dtype=torch.float64)
delta = torch.randn(4, 8, generator=generator, dtype=torch.float64)
bias = torch.randn(4, generator=generator, dtype=torch.float64)
torch.testing.assert_close(
    F.linear(a, w + delta, bias),
    F.linear(a, w, bias) + F.linear(a, delta),
    rtol=1e-12,
    atol=1e-12,
)
print("Float64 linear identity passed")


Float64 linear identity passed


In [5]:
# NBVAL_IGNORE_OUTPUT
load_started = time.perf_counter()
available_gib = psutil.virtual_memory().available / 2**30
print(f"Available RAM before loading: {available_gib:.1f} GiB")
bridge = TransformerBridge.boot_transformers(
    str(snapshot),
    device="cpu",
    dtype=torch.float32,
).eval()
layers = bridge.original_model.model.language_model.layers
for layer, edit in edits.items():
    parameter = layers[layer].mlp.down_proj.weight
    assert parameter.dtype == torch.float32
    assert torch.equal(parameter.detach(), edit["base"])
timings["model_load_seconds"] = time.perf_counter() - load_started
print("Pinned checkpoint loaded on CPU in fp32")


Available RAM before loading: 6.1 GiB


Pinned checkpoint loaded on CPU in fp32


In [6]:
# NBVAL_IGNORE_OUTPUT
@contextmanager
def replaced_weights(replacements):
    with ExitStack() as stack:
        for layer, value in replacements.items():
            stack.enter_context(
                temporarily_swap_parameter(layers[layer].mlp.down_proj.weight, value)
            )
        yield


def difference_hooks(use_fit=False):
    pending, hooks = {}, []
    for layer, edit in edits.items():
        columns = edit["columns"]
        difference = edit["fitted_delta"] if use_fit else edit["delta"]

        def keep(activation, hook, layer=layer, columns=columns):
            pending[layer] = activation[..., columns]

        def change(output, hook, layer=layer, difference=difference):
            correction = F.linear(pending.pop(layer), difference)
            return output + correction

        hooks.extend(
            [
                (f"blocks.{layer}.mlp.hook_post", keep),
                (f"blocks.{layer}.mlp.hook_out", change),
            ]
        )
    return hooks


@torch.no_grad()
def forward_capture(tokens, hooks=()):
    captured = {}

    def capture(activation, hook):
        captured[hook.name] = activation.detach().clone()

    names = [f"blocks.{layer}.mlp.hook_out" for layer in LAYERS]
    with bridge.hooks(fwd_hooks=list(hooks) + [(name, capture) for name in names]):
        logits = bridge(tokens)
    return logits, captured


def error_summary(actual, expected):
    difference = (actual - expected).abs()
    return {"mean_abs": difference.mean().item(), "max_abs": difference.max().item()}


texts = [
    "First compute the value, then check the result. " * 4,
    "Let x be the unknown. Substitute x into the equation. " * 4,
]
inputs = [
    bridge.tokenizer(text, add_special_tokens=False, return_tensors="pt").input_ids[
        :, :MAX_TOKENS
    ]
    for text in texts
]
validation_started = time.perf_counter()
comparisons = []
for input_number, tokens in enumerate(inputs):
    baseline, _ = forward_capture(tokens)
    with replaced_weights({layer: e["edited"] for layer, e in edits.items()}):
        swapped, swapped_components = forward_capture(tokens)
    assert not torch.allclose(swapped, baseline, rtol=1e-4, atol=1e-4)
    exact, exact_components = forward_capture(tokens, difference_hooks())
    fitted, _ = forward_capture(tokens, difference_hooks(use_fit=True))
    assert torch.isfinite(fitted).all()
    for name in swapped_components:
        comparisons.append(
            {
                "input": input_number,
                "tokens": tokens.shape[1],
                "comparison": f"exact hook vs swapped: {name}",
                **error_summary(exact_components[name], swapped_components[name]),
            }
        )
        torch.testing.assert_close(
            exact_components[name], swapped_components[name], rtol=1e-4, atol=1e-4
        )
    torch.testing.assert_close(exact, swapped, rtol=1e-4, atol=1e-4)
    for label, actual, expected in [
        ("exact hook vs swapped", exact, swapped),
        ("fitted hook vs exact hook", fitted, exact),
        ("edited vs baseline", swapped, baseline),
    ]:
        comparisons.append(
            {
                "input": input_number,
                "tokens": tokens.shape[1],
                "comparison": label,
                **error_summary(actual, expected),
            }
        )

    def token_nll(logits):
        return F.cross_entropy(logits[0, :-1].float(), tokens[0, 1:], reduction="none")

    swapped_nll, exact_nll = token_nll(swapped), token_nll(exact)
    torch.testing.assert_close(exact_nll, swapped_nll, rtol=1e-4, atol=1e-4)
    comparisons.append(
        {
            "input": input_number,
            "tokens": tokens.shape[1],
            "comparison": "exact hook vs swapped: token NLL",
            **error_summary(exact_nll, swapped_nll),
        }
    )
    restored, _ = forward_capture(tokens)
    torch.testing.assert_close(restored, baseline, rtol=0, atol=0)

# Exercise restoration when the experiment body raises.
try:
    with replaced_weights({layer: e["edited"] for layer, e in edits.items()}):
        raise ValueError("restoration probe")
except ValueError as error:
    assert str(error) == "restoration probe"
for layer, edit in edits.items():
    assert torch.equal(layers[layer].mlp.down_proj.weight.detach(), edit["base"])
timings["forward_validation_seconds"] = time.perf_counter() - validation_started
display(pd.DataFrame(comparisons))


,input,tokens,comparison,mean_abs,max_abs
0,0,32,exact hook vs swapped: blocks.15.mlp.hook_out,8.780711e-09,3.054738e-07
1,0,32,exact hook vs swapped: blocks.19.mlp.hook_out,2.741603e-08,2.160668e-07
2,0,32,exact hook vs swapped,1.104146e-06,1.049042e-05
3,0,32,fitted hook vs exact hook,3.838236e-04,3.693104e-03
4,0,32,edited vs baseline,7.012547e-01,9.933302e+00
5,0,32,exact hook vs swapped: token NLL,9.273570e-07,7.629395e-06
6,1,32,exact hook vs swapped: blocks.15.mlp.hook_out,8.276834e-09,3.278255e-07
7,1,32,exact hook vs swapped: blocks.19.mlp.hook_out,2.772856e-08,4.470348e-07
8,1,32,exact hook vs swapped,1.139451e-06,9.059906e-06
9,1,32,fitted hook vs exact hook,3.841585e-04,4.105568e-03


The assertion applies to the exact difference, not the fitted rank-one edit. The error table reports both comparisons. The restored baseline also matches after normal completion and an exception.

## 5. Recorded larger-window experiment

A timed local run on **2026-11-05** used an **NVIDIA GeForce RTX 5060 Laptop GPU**, bf16, PyTorch 2.11.0+cu128, Transformers 5.13.0 and TransformerLens 837fb727. Each condition used 40 long base-model transcript windows of 1,024 tokens and their early windows. A copy token ends an 8-gram already seen in that window. Values are equal-weight means over windows, in nats/token.

The 13-condition run took **443.5 seconds total**, including **408.7 seconds for the sweep**; peak allocated CUDA memory was **2.63 GiB**. The six table conditions and seven additional coefficients used 1,040 forward passes. Selected problem IDs, source-file hashes, versions, condition timings and summary values are included in this notebook's `weight_edit_audit_run` metadata.

| Intervention | Copy-token ΔNLL, late windows | Other-token ΔNLL, late windows | All-token ΔNLL, early windows |
|---|---:|---:|---:|
| Fitted directional hook, α≈6, selected 64 columns | +0.0600 | +0.0067 | +0.0091 |
| α=1, same columns | +0.0012 | −0.0006 | +0.0003 |
| α=1, whole MLP output | +0.0017 | +0.0052 | +0.0014 |
| Random direction, same α and columns | +0.0000 | +0.0003 | −0.0000 |
| Random columns, same α and direction | −0.0001 | −0.0012 | +0.0005 |

The same sample set gives this coefficient response for repeated tokens in late windows:

| α | 0.5 | 1 | 2 | 3 | 4 | 6 | 8 | 10 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| ΔNLL | +0.0005 | +0.0012 | +0.0030 | +0.0063 | +0.0126 | +0.0599 | +0.2480 | +0.5900 |

Controls share α and column count; their update norms and activation displacements are unmatched. These measurements use isolated teacher-forced windows, whereas the authors' benchmark evaluates generated answers from full problem prompts. The static experiment is separate from the executable CPU agreement check above.

## 6. Reusing the audit

For another edit, retain the original-value checks, compute the exact difference first, and compare it with a structured approximation. A coefficient sweep then answers a more precise question: how does that measured update affect the selected inputs? Keep sampling IDs, per-window results, device, versions and timings with the experiment.


In [7]:
# NBVAL_IGNORE_OUTPUT
timings["total_seconds"] = time.perf_counter() - notebook_started
print(timings)
process = psutil.Process()
memory = process.memory_info()
peak_bytes = getattr(memory, "peak_wset", getattr(memory, "rss", 0))
print(
    {
        "process_memory_GiB": peak_bytes / 2**30,
        "memory_measure": "Windows peak working set"
        if hasattr(memory, "peak_wset")
        else "current RSS",
    }
)


{'artifacts_and_checkpoint_seconds': 0.10814539999955741, 'rank_one_fit_seconds': 0.0068721000006917166, 'model_load_seconds': 2.183606900000086, 'forward_validation_seconds': 13.068054299999858, 'total_seconds': 21.910111600000164}
{'process_memory_GiB': 5.9306640625, 'memory_measure': 'Windows peak working set'}
